# Qwen P2 grid: MULTICLASS grid-probe balanced accuracy by grid-loudness decile

The multiclass twin of `./probe_accuracy_by_loudness_decile.ipynb` (the binary one-vs-rest probes), cell for cell:
same held-out rows, same cells, same deciles, same bootstrap, same figures' layout. Held-out 72 (70 gathered). One row per
(trajectory, step, reasoning token): **every** token, so no lens chose the rows. There are 6 multiclass probes:
3 selection arms (jlens / logitlens / random) x {lr, mlp}, all at layer 27 (the Qwen direction profile's J-lens argmax), each predicting a
cell's symbol among A # G _ and `+` padding.

**Input.** `score_probes_per_token.py --probe-type grid_multiclass`, run as recorded in cell 1 (`SCORED_WITH`).
Each row carries the token's cells counted by true class (`n_true_{c}`) and, per probe, how many of each class
it got right (`{probe}_correct_{c}`). It also carries both lenses' **grid** loudness at L27
(`{lens}_grid_logmass_L27`, log P(any pruned-vocabulary grid word)), read from `qwen_p2_heldout_grid_lens`.

**The cells are prepare's.** `grid_multiclass` draws them with `prepare_activations_for_probing._grid_cell_payload`
using the same seed/pad/cap as `prepared/qwen_p2_grid_heldout`, exactly as `grid_binary` does. So these are the rows
`eval_multiclass_probes_heldout.sh` scored, and **cell 2 must reproduce those 6 JSONs**. If it does not, stop.

**Statistics are the repo's.** `stats.bal_acc_from_counts` pools the per-class counts over a bin before dividing:
the mean of pooled recall over the **real** classes A # G _. Padding is excluded: a cell outside the grid says nothing
about the grid, and it is the easy majority of a small grid's cells. So chance is **0.25**, and a token's 25 cells carry
25 cells' weight. The 95% bands resample **trajectory names**, never rows. Deciles come from `stats.qbin` and are re-cut
per lens. `columns.axis_label` names every axis.

**Caveats.** Most grid mass is AXIS (row/column words; `data/jlens/README.md`), so "grid-loud" is largely
"coordinate-loud". A and G are one cell per grid, so their recalls rest on few cells per decile. **Qwen lens outputs before the 2026-09-23 norm fix used the wrong final norm** (`wrappers/qwen_analysis/norm_fix/`): point `LENS_DIR` at the corrected tree before reading these as final.

## 1. Load

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from telos_interp.grid_utils import CELL_SYMBOL_TO_ID
from telos_interp.loudness_analysis import columns as cols
from telos_interp.loudness_analysis import stats

# ---- what this run is ---------------------------------------------------------------
TABLE = Path("/workspace/results/qwen_p2_grid/heldout_multiclass/per_token_scores.csv")
EVAL_JSONS = Path("/workspace/results/qwen_p2_grid/heldout_multiclass")  # eval_multiclass_probes_heldout.sh's output
FIG_DIR = TABLE.parent / "figures"
SIGNAL_JSON = Path("/workspace/repo/interp/data/jlens/qwen/grid_tokens_pruned_qwen3-6-35b-a3b.json")
LAYER = 27
SIGNAL = "grid"
LENSES = ("jlens", "logitlens")
N_DECILES = 10
N_BOOT = 300
SEED = 42
EXCLUDE_RADIUS = 2  # verbalisation control: drop grid words and their +-2 neighbours

SCORED_WITH = """
uv run --extra gpu python telos_interp/loudness_analysis/score_probes_per_token.py \\
    --probe-type grid_multiclass --probe /workspace/probes/qwen_p2_grid/qwen_p2_grid_<arm>_multiclass_l27_<lr|mlp>.pt (x6) \\
    --activations-dir /workspace/activations/qwen_p2_heldout \\
    --lens-dir /workspace/activations/qwen_p2_heldout_grid_lens \\
    --trajectories-dir /workspace/trajectories/qwen3.6-35b/replayed_single_step/heldout_72 \\
    --signal-json /workspace/repo/interp/data/jlens/qwen/grid_tokens_pruned_qwen3-6-35b-a3b.json --signal-name grid \\
    --layer 27 --pad-to-size 15 --max-cells 25 --seed 42 \\
    --cache-activations --cache-dir /workspace/results/qwen_p2_local_belief/heldout/_act_cache \\
    --read-threads 16 --device cuda --out /workspace/results/qwen_p2_grid/heldout_multiclass/per_token_scores.csv
"""

FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

# keep_default_na=False: a decoded token can literally be "NA". Numeric columns are coerced below.
df = pd.read_csv(TABLE, keep_default_na=False, na_values=[""], low_memory=False)
print(f"{len(df):,} token rows, {df['name'].nunique()} trajectories")

# ---- which probes are in the table: discovered from the columns, never hard-coded --------
PROBE_KEYS = sorted(c[: -len("_n_correct")] for c in df.columns if c.endswith("_n_correct"))
KEY_RE = re.compile(r"_(?P<arm>jlens|logitlens|random)_multiclass_l\d+_(?P<mt>lr|mlp)$")
META = {k: KEY_RE.search(k).groupdict() for k in PROBE_KEYS}
ARMS = ["jlens", "logitlens", "random"]
MTYPES = ["lr", "mlp"]
# The real cell classes, by id; padding ('+') is left out of every statistic.
REAL = {CELL_SYMBOL_TO_ID[s]: name for s, name in (("_", "empty"), ("#", "wall"), ("A", "agent"), ("G", "goal"))}
CLASSES = list(REAL.values())
CHANCE = 1.0 / len(REAL)
COUNT_COLS = [f"n_true_{c}" for c in REAL] + [f"{k}_correct_{c}" for k in PROBE_KEYS for c in REAL]
df[COUNT_COLS] = df[COUNT_COLS].apply(pd.to_numeric).astype(np.int64)
COLOR = dict(zip(ARMS, sns.color_palette("colorblind", len(ARMS)), strict=True))
print(f"{len(PROBE_KEYS)} probes: {sorted({m['arm'] for m in META.values()})} x {sorted({m['mt'] for m in META.values()})}")

LOUD = {lens: cols.resolve(df.columns, lens, SIGNAL, LAYER) for lens in LENSES}
for lens, c in LOUD.items():
    df[c] = pd.to_numeric(df[c], errors="coerce")
    print(f"  {lens}: {c}  ({df[c].notna().mean():.1%} of rows have a mass cell)")

## 2. Check: the pooled numbers must reproduce the held-out evaluator

Pool every row, then compare with `eval_grid_probe.py`'s `global` block for the same probe on the same manifest.
Same cells means the **ground truth must match exactly**: every real class's support, and the total cell count.
The **verdicts** may differ by a handful of cells out of ~29M, because the GPU rounds batches of different
shapes slightly differently and a near-tie between two classes can fall either way.

In [ ]:
check = []
for k in PROBE_KEYS:
    m = META[k]
    ba, _ = stats.bal_acc_from_counts(df, k, REAL)
    ref = json.loads((EVAL_JSONS / f"{m['arm']}_multiclass_{m['mt']}.json").read_text())["global"]
    pc = ref["per_class"]
    same_truth = all(int(df[f"n_true_{c}"].sum()) == pc[str(c)]["support"] for c in REAL) and int(
        df["n_cells"].astype(int).sum()
    ) == ref["n_rows"]
    flipped = sum(abs(int(df[f"{k}_correct_{c}"].sum()) - pc[str(c)]["correct"]) for c in REAL)
    check.append({**m, "bal_acc": ba, "eval_json": ref["balanced_accuracy_no_padding"],
                  "abs_diff": abs(ba - ref["balanced_accuracy_no_padding"]), "same_truth": same_truth, "cells_flipped": flipped})
check = pd.DataFrame(check)
print(f"ground truth identical for all {len(check)} probes: {bool(check.same_truth.all())}")
print(f"verdicts: max {check.cells_flipped.max()} cell(s) of {df.n_cells.astype(int).sum():,} differ; max |bal_acc diff| = {check.abs_diff.max():.1e}")
assert check.same_truth.all(), "different ROWS from the evaluator -- stop here"
assert check.cells_flipped.max() <= 10 and check.abs_diff.max() < 1e-5, "verdicts differ beyond rounding jitter -- stop here"
check.pivot_table(index="mt", columns="arm", values="bal_acc").round(4)

## 3. Balanced accuracy by loudness decile, per lens

Counts are summed per (trajectory, decile) first. The bootstrap then resamples trajectory names over that small
frame instead of over every row, which gives the same statistic much faster. Every per-class recall is
bootstrapped with the same draws.

In [ ]:
def with_decile(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    d = frame[np.isfinite(frame[LOUD[lens]])].copy()
    d["decile"] = stats.qbin(d[LOUD[lens]], N_DECILES, labels=False).astype(int) + 1
    return d


def pooled(counts: pd.DataFrame, k: str) -> tuple[float, dict]:
    return stats.bal_acc_from_counts(counts, k, REAL)


def decile_table(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    d = with_decile(frame, lens)
    agg = d.groupby(["decile", "name"])[COUNT_COLS].sum()
    meta = d.groupby("decile").agg(n_tokens=("name", "size"), n_traj=("name", "nunique"), mean_logmass=(LOUD[lens], "mean"))
    rng = np.random.default_rng(SEED)
    names = d["name"].unique()
    draws = [rng.choice(names, size=len(names), replace=True) for _ in range(N_BOOT)]
    rows = []
    for dec, g in agg.groupby(level="decile"):
        g = g.droplevel("decile")
        boot = [g.reindex(pick).fillna(0) for pick in draws]
        for k in PROBE_KEYS:
            ba, rec = pooled(g, k)
            bs = [pooled(b, k) for b in boot]
            row = {"lens": lens, "decile": int(dec), **META[k], "probe_key": k, "bal_acc": ba,
                   "lo": np.nanpercentile([b[0] for b in bs], 2.5), "hi": np.nanpercentile([b[0] for b in bs], 97.5)}
            for c, name in REAL.items():
                r = [b[1].get(c, np.nan) for b in bs]
                row |= {f"recall_{name}": rec.get(c, np.nan),
                        f"recall_{name}_lo": np.nanpercentile(r, 2.5), f"recall_{name}_hi": np.nanpercentile(r, 97.5)}
            rows.append(row | meta.loc[dec].to_dict())
    return pd.DataFrame(rows)


BY_DECILE = pd.concat([decile_table(df, lens) for lens in LENSES], ignore_index=True)
BY_DECILE.to_csv(TABLE.parent / "accuracy_by_grid_loudness_decile.csv", index=False)
print(BY_DECILE[BY_DECILE.lens == "jlens"].drop_duplicates("decile")[["decile", "n_tokens", "n_traj", "mean_logmass"]].to_string(index=False))

### Loudest minus quietest decile, with a trajectory-clustered 95% CI

The same resampled names are used for both ends, so the CI is on the **gap**, not two independent bands.

In [ ]:
def gap_table(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    d = with_decile(frame, lens)
    ends = d[d.decile.isin([1, N_DECILES])].groupby(["decile", "name"])[COUNT_COLS].sum()
    lo_g, hi_g = ends.loc[1], ends.loc[N_DECILES]
    rng = np.random.default_rng(SEED)
    names = d["name"].unique()
    draws = [rng.choice(names, size=len(names), replace=True) for _ in range(N_BOOT)]
    rows = []
    for k in PROBE_KEYS:
        gap = pooled(hi_g, k)[0] - pooled(lo_g, k)[0]
        bs = np.array([pooled(hi_g.reindex(p).fillna(0), k)[0] - pooled(lo_g.reindex(p).fillna(0), k)[0] for p in draws])
        rows.append({"lens": lens, **META[k], "gap_top_minus_bottom": gap,
                     "lo": np.nanpercentile(bs, 2.5), "hi": np.nanpercentile(bs, 97.5)})
    return pd.DataFrame(rows)


GAPS = pd.concat([gap_table(df, lens) for lens in LENSES], ignore_index=True)
GAPS.to_csv(TABLE.parent / "grid_loudness_gap_top_minus_bottom.csv", index=False)
GAPS.pivot_table(index=["lens", "mt"], columns="arm", values="gap_top_minus_bottom").round(4)

## 4. Figures

Two per lens. **Balanced accuracy**: one panel per probe family (lr, mlp), the three arms overlaid. **Per-class
recall**: one panel per (probe family, class), the binary notebook's layout, so each class can be read against its
one-vs-rest twin. Every point in a panel is the same tokens and the same cells; only the probe's training
selection differs.

In [ ]:
def draw(tab: pd.DataFrame, lens: str, title: str, fname: str) -> None:
    fig, axes = plt.subplots(1, len(MTYPES), figsize=(5.2 * len(MTYPES), 4.0), sharey=True)
    for c, mt in enumerate(MTYPES):
        ax = axes[c]
        for arm in ARMS:
            t = tab[(tab.lens == lens) & (tab.arm == arm) & (tab.mt == mt)].sort_values("decile")
            ax.plot(t.decile, t.bal_acc, marker="o", ms=3.5, lw=1.6, color=COLOR[arm], label=arm)
            ax.fill_between(t.decile, t.lo, t.hi, color=COLOR[arm], alpha=0.15, lw=0)
        ax.axhline(CHANCE, ls=":", lw=1, color="0.4")
        ax.set_title(f"multiclass / {mt}", fontsize=11)
        if c == 0:
            ax.set_ylabel("balanced accuracy, A # G _\n(cells pooled)")
        ax.set_xlabel(f"{cols.axis_label(lens, SIGNAL, LAYER)} decile\n(1 = quietest)")
        ax.set_xticks(range(1, N_DECILES + 1))
    axes[0].legend(title="probe trained on", fontsize=8, title_fontsize=8, loc="best")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{fname}.png", dpi=150)
    print(f"-> {FIG_DIR / f'{fname}.png'}")
    plt.show()


def draw_per_class(tab: pd.DataFrame, lens: str, title: str, fname: str) -> None:
    fig, axes = plt.subplots(len(MTYPES), len(CLASSES), figsize=(4.2 * len(CLASSES), 3.6 * len(MTYPES)), sharex=True)
    for r, mt in enumerate(MTYPES):
        for c, cls in enumerate(CLASSES):
            ax = axes[r][c]
            for arm in ARMS:
                t = tab[(tab.lens == lens) & (tab.arm == arm) & (tab.mt == mt)].sort_values("decile")
                ax.plot(t.decile, t[f"recall_{cls}"], marker="o", ms=3.5, lw=1.6, color=COLOR[arm], label=arm)
                ax.fill_between(t.decile, t[f"recall_{cls}_lo"], t[f"recall_{cls}_hi"], color=COLOR[arm], alpha=0.15, lw=0)
            ax.set_title(f"{cls} / {mt}", fontsize=11)
            if c == 0:
                ax.set_ylabel("recall\n(cells pooled)")
            if r == len(MTYPES) - 1:
                ax.set_xlabel(f"{cols.axis_label(lens, SIGNAL, LAYER)} decile\n(1 = quietest)")
            ax.set_xticks(range(1, N_DECILES + 1))
    axes[0][0].legend(title="probe trained on", fontsize=8, title_fontsize=8, loc="best")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{fname}.png", dpi=150)
    print(f"-> {FIG_DIR / f'{fname}.png'}")
    plt.show()


draw(BY_DECILE, "jlens", "Qwen held-out 70, every reasoning token: multiclass grid probes by J-lens grid loudness", "grid_decile_jlens")
draw_per_class(BY_DECILE, "jlens", "Qwen held-out 70: per-class recall by J-lens grid loudness", "grid_decile_jlens_per_class")

In [ ]:
draw(BY_DECILE, "logitlens", "Qwen held-out 70, every reasoning token: multiclass grid probes by logit-lens grid loudness", "grid_decile_logitlens")
draw_per_class(BY_DECILE, "logitlens", "Qwen held-out 70: per-class recall by logit-lens grid loudness", "grid_decile_logitlens_per_class")

## 5. Verbalisation control

The lens predicts the *next* tokens, so a token just before ` row` is loud without being a grid word itself.
This drops every token that **is** a pruned-vocabulary grid word, plus its +-`EXCLUDE_RADIUS` neighbours
within the same (trajectory, step), then redraws the J-lens panels. The table's tokens are raw byte-level BPE
(`Ġrow`) and the vocabulary is decoded text (` row`), so tokens are normalised before matching, as
`rollouts/eval_local_belief.py` does.

In [ ]:
vocab = {t for lst in json.loads(SIGNAL_JSON.read_text()).values() for t in lst}
flag = df["token"].str.replace("Ġ", " ", regex=False).str.replace("Ċ", "\n", regex=False).isin(vocab)
drop = flag.copy()
grouped = flag.groupby([df["name"], df["step"]])
for s in range(1, EXCLUDE_RADIUS + 1):
    drop |= grouped.shift(s, fill_value=False) | grouped.shift(-s, fill_value=False)
quiet = df[~drop]
print(f"grid words: {flag.mean():.2%} of tokens; dropped with radius {EXCLUDE_RADIUS}: {drop.mean():.2%} -> {len(quiet):,} rows")

BY_DECILE_NOVERB = decile_table(quiet, "jlens")
BY_DECILE_NOVERB.to_csv(TABLE.parent / f"accuracy_by_grid_loudness_decile_no_grid_words_r{EXCLUDE_RADIUS}.csv", index=False)
draw(BY_DECILE_NOVERB, "jlens", f"...excluding grid words and their +-{EXCLUDE_RADIUS} neighbours (J-lens)", f"grid_decile_jlens_no_grid_words_r{EXCLUDE_RADIUS}")
draw_per_class(BY_DECILE_NOVERB, "jlens", f"...per class, excluding grid words and their +-{EXCLUDE_RADIUS} neighbours (J-lens)", f"grid_decile_jlens_per_class_no_grid_words_r{EXCLUDE_RADIUS}")
gap_table(quiet, "jlens").pivot_table(index="mt", columns="arm", values="gap_top_minus_bottom").round(4)